Run 
> pip install -r requirements.txt

In [28]:
from pdf2image import convert_from_path
import pytesseract
import numpy as np
import cv2
import json
import os
import json


## pdf_to_images
Splits pdf into multiple individual page images of jpeg format and saves them to storage as jpeg
##### Args:
- pdf_path (str): Filepath to pdf
- dpi (int): Image resolution, 300 by default
##### Returns:
A list containing paths to each individual image

In [29]:

def pdf_to_images(pdf_path, dpi=300):
    """
    Splits pdf into multiple individual page images of jpeg format and saves them to storage as jpeg

    Args:
        pdf_path (str): Filepath to pdf
        dpi (int): Image resolution, 300 by default

    Returns:
        A list containing paths to each individual image
    """
    # 300 DPI is standard for OCR tasks
    pages = convert_from_path(pdf_path, dpi=dpi)
    image_paths = []
    
    for i, page in enumerate(pages):
        filename = f'page_{i}.jpg'
        page.save(filename, 'JPEG')
        image_paths.append(filename)
        
    return image_paths
image_paths = pdf_to_images('sample.pdf')

## stitch_images_vertically
Vertically stitches a set of images together and saves it to storage.
##### Args:
- image_paths (list of str): List containing filepaths to images to be stitched together.
- filename (str): Name of saved file. Defalut value is "combined_paper.jpg".
##### Returns:
This function returns a NumPy array (numpy.ndarray) representing the final vertically stitched image.
Specifically, it returns the long_image object created by cv2.vconcat(), which holds the pixel data of all the combined images stored in your computer's RAM.

In [30]:

def stitch_images_vertically(image_paths, filename='combined_paper.jpg'):
    """
    Vertically stitches a set of images together and saves it to storage.

    Args:
        image_paths (list of str): List containing filepaths to images to be stitched together.
        filename (str): Name of saved file. Defalut value is "combined_paper.jpg".

    Returns:
        This function returns a NumPy array (numpy.ndarray) representing the final vertically stitched image.
        Specifically, it returns the long_image object created by cv2.vconcat(), 
        which holds the pixel data of all the combined images stored in your computer's RAM.
    """
    images = [cv2.imread(img) for img in image_paths]
    
    # Ensure all images have the same width before concatenating
    max_width = max(img.shape[1] for img in images)
    resized_images = []
    for img in images:
        if img.shape[1] != max_width:
            img = cv2.resize(img, (max_width, int(img.shape[0] * max_width / img.shape[1])))
        resized_images.append(img)
        
    # Vertically concatenate
    long_image = cv2.vconcat(resized_images)
    cv2.imwrite(filename, long_image)
    return long_image
stitch_images_vertically(image_paths[2::2])

array([[[255, 255, 255],
        [255, 255, 255],
        [255, 255, 255],
        ...,
        [255, 255, 255],
        [255, 255, 255],
        [255, 255, 255]],

       [[255, 255, 255],
        [255, 255, 255],
        [255, 255, 255],
        ...,
        [255, 255, 255],
        [255, 255, 255],
        [255, 255, 255]],

       [[255, 255, 255],
        [255, 255, 255],
        [255, 255, 255],
        ...,
        [255, 255, 255],
        [255, 255, 255],
        [255, 255, 255]],

       ...,

       [[255, 255, 255],
        [255, 255, 255],
        [255, 255, 255],
        ...,
        [255, 255, 255],
        [255, 255, 255],
        [255, 255, 255]],

       [[255, 255, 255],
        [255, 255, 255],
        [255, 255, 255],
        ...,
        [255, 255, 255],
        [255, 255, 255],
        [255, 255, 255]],

       [[255, 255, 255],
        [255, 255, 255],
        [255, 255, 255],
        ...,
        [255, 255, 255],
        [255, 255, 255],
        [255, 255, 255]]

## split_into_sections
Splits a long stitched question paper image into separate section images 
based on matched keywords or headings found via OCR.
##### Args:
- long_image (numpy.ndarray): The OpenCV image array of the full combined paper.
- selection_texts (list of str): A list of target strings to look for to trigger a vertical split.
##### Returns:
list of numpy.ndarray: A list containing individual sliced section image arrays.

In [31]:
def split_into_sections(long_image, selection_texts):
    """
    Splits a long stitched question paper image into separate section images 
    based on matched keywords or headings found via OCR.

    Args:
        long_image (numpy.ndarray): The OpenCV image array of the full combined paper.
        selection_texts (list of str): A list of target strings to look for to trigger a vertical split.

    Returns:
        list of numpy.ndarray: A list containing individual sliced section image arrays.
    """


    # Convert to grayscale
    gray = cv2.cvtColor(long_image, cv2.COLOR_BGR2GRAY)
    
    # Get bounding boxes for all text elements
    custom_config = r'--oem 3 --psm 6'
    data = pytesseract.image_to_data(gray, config=custom_config, output_type=pytesseract.Output.DICT)
    
    section_images = []
    split_y_coordinates = []
    
    for i in range(len(data['text'])):
        text = data['text'][i].strip()
        # Look for patterns like "Q1", "1.", "Section A"
        # if text.startswith('Q') and text[1:].isdigit() or text.lower() == 'section':
        if text.lower() in selection_texts:
            y = data['top'][i]
            print(y)
            # Add some padding above the text
            split_y_coordinates.append(max(0, y - 55)) 
            
    split_y_coordinates.append(long_image.shape[0]) # Add bottom of image
    
    # Slice the image based on coordinates
    for i in range(len(split_y_coordinates) - 1):
        y_start = split_y_coordinates[i]
        y_end = split_y_coordinates[i+1]
        
        # Avoid slicing empty spaces
        if y_end - y_start > 50: 
            s_img = long_image[y_start:y_end, 0:long_image.shape[1]]
            section_images.append(s_img)
    return section_images

## extract_figures
Extracts large diagrams, photographs, and figures from a question image while 
filtering out sparse math equations, plain text blocks, and thin divider lines.

The function applies Otsu's thresholding and minimal morphological dilation to 
isolate distinct image contours. It relies on a combination of minimum area thresholds, 
aspect ratios, and ink density (fill ratio) to distinguish dense graphical media 
from sparse text or mathematical symbols.
##### Args:
- image (numpy.ndarray): The BGR OpenCV image array containing a single question.
- q_number (int): The current question number (used for file naming output).
##### Returns:
list of dict: A list of dictionaries for each detected figure. Each dictionary contains:
- 'path' (str): The local file path where the figure image was saved.
- 'box' (tuple of int): The bounding box tuple formatted as (x, y, width, height).

In [32]:
def extract_figures(image, q_number):

    """
    Extracts large diagrams, photographs, and figures from a question image while 
    filtering out sparse math equations, plain text blocks, and thin divider lines.

    The function applies Otsu's thresholding and minimal morphological dilation to 
    isolate distinct image contours. It relies on a combination of minimum area thresholds, 
    aspect ratios, and ink density (fill ratio) to distinguish dense graphical media 
    from sparse text or mathematical symbols.

    Args:
        image (numpy.ndarray): The BGR OpenCV image array containing a single question.
        q_number (int): The current question number (used for file naming output).

    Returns:
        list of dict: A list of dictionaries for each detected figure. Each dictionary contains:
            - 'path' (str): The local file path where the figure image was saved.
            - 'box' (tuple of int): The bounding box tuple formatted as (x, y, width, height).

    Raises:
        cv2.error: If the input image is invalid or cannot be converted to grayscale.
    """
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    
    # 1. Use Otsu's thresholding to cleanly separate dark ink/photos from the white background
    _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    
    # 2. FIX: Drastically reduce dilation. 
    # A smaller 5x5 kernel connects parts of the photo/diagram's border 
    # without expanding outward into the surrounding question text.
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (5, 5))
    dilated = cv2.dilate(thresh, kernel, iterations=1)
    
    contours, _ = cv2.findContours(dilated, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    figures = []
    for i, c in enumerate(contours):
        x, y, w, h = cv2.boundingRect(c)
        area = w * h
        
        # Ensure it is a large block and has substantial height/width
        if 40000 < area < 3000000 and w > 100 and h > 100:
            
            # 3. FIX: The "Ink Density" or Fill Ratio check.
            # We count how many "dark" pixels exist inside this bounding box.
            roi_thresh = thresh[y:y+h, x:x+w]
            ink_pixels = cv2.countNonZero(roi_thresh)
            fill_ratio = ink_pixels / float(area)
            
            aspect_ratio = w / float(h)
            
            # Require at least 20% of the bounding box to be actual dark pixels. 
            # (Math equations usually hover around 5-12% fill ratio; photos easily pass 20%).
            if 0.2 < aspect_ratio < 5.0 and fill_ratio > 0.20:
                
                # Optional: Add a small 2-pixel padding to the crop so the border isn't clipped
                pad = 2
                y_start = max(0, y - pad)
                y_end = min(image.shape[0], y + h + pad)
                x_start = max(0, x - pad)
                x_end = min(image.shape[1], x + w + pad)
                
                fig_img = image[y_start:y_end, x_start:x_end]
                fig_path = f'q{q_number}_figure_{i}.jpg'
                
                cv2.imwrite(fig_path, fig_img)
                figures.append({'path': fig_path, 'box': (x, y, w, h)})
                
    return figures

## extract_text_without_figures
Masks out detected figures/diagrams from a question image with solid white rectangles 
and extracts the remaining text using Tesseract OCR.
##### Args:
- image (numpy.ndarray): The BGR OpenCV image array of the question.
- figures (list of dict): A list of dictionaries containing detected figure metadata. Each dictionary must include a 'box' key with a tuple formatted as (x, y, width, height).
##### Returns:
 str: The extracted, cleaned text string with figure areas removed and outer whitespace trimmed.

In [33]:
def extract_text_without_figures(image, figures):

    """
    Masks out detected figures/diagrams from a question image with solid white rectangles 
    and extracts the remaining text using Tesseract OCR.

    Args:
        image (numpy.ndarray): The BGR OpenCV image array of the question.
        figures (list of dict): A list of dictionaries containing detected figure metadata. 
                                Each dictionary must include a 'box' key with a tuple 
                                formatted as (x, y, width, height).

    Returns:
        str: The extracted, cleaned text string with figure areas removed and outer 
             whitespace trimmed.

    Raises:
        KeyError: If any dictionary in `figures` lacks the 'box' key.
        cv2.error: If image copying or drawing fails due to invalid image arrays.
    """
    cleaned_image = image.copy()
    
    # Mask out the figures with white rectangles
    for index, fig in enumerate(figures):
        x, y, w, h = fig['box']
        cv2.rectangle(cleaned_image, (x, y), (x+w, y+h), (255, 255, 255), -1)
        cv2.imwrite(f'{index}.jpg', cleaned_image)

        
    # Extract Text
    text = pytesseract.image_to_string(cleaned_image)
    return text.strip()

In [34]:

def process_question_paper(long_image_path):
    # 1. Load the long combined image
    long_image = cv2.imread(long_image_path)
    if long_image is None:
        raise ValueError("Could not load image. Check the path.")

    # Create a directory to store the outputs to keep things organized
    output_dir = "extracted_data"
    os.makedirs(output_dir, exist_ok=True)

    # 2. Split the long image into individual questions
    print("Splitting into individual questions...")
    section_images = split_into_sections(long_image, [f'{i}.' for i in range(1,15)])
    
    pipeline_results = []

    # 3. Iterate through the returned list of question images
    for index, s_img in enumerate(section_images):
        q_number = index + 1  # Start numbering from 1 instead of 0
        
        # --- SAVE THE QUESTION IMAGE ---
        q_image_path = os.path.join(output_dir, f'question_{q_number}.jpg')
        cv2.imwrite(q_image_path, s_img)
        print(f"Saved: {q_image_path}")

        # --- EXTRACT FIGURES FOR THIS SPECIFIC QUESTION ---
        # Note: We are passing the OpenCV array (s_img), NOT the saved file path
        figures_data = extract_figures(s_img, q_number)
        q_text = extract_text_without_figures(s_img, figures_data)
        
        
        # Optional: Move the saved figures into the output directory to keep the root clean
        for fig in figures_data:
            old_path = fig['path']
            new_path = os.path.join(output_dir, old_path)
            os.rename(old_path, new_path)
            fig['path'] = new_path

        # Collect data for your final JSON dataset
        pipeline_results.append({
            "question_id": q_number,
            "question_image_path": q_image_path,
            "extracted_figures": figures_data,
            "question_text": q_text
        })

    return pipeline_results

In [35]:

def process_and_save_to_json(section_images, output_filename='dataset.json'):
    dataset = []
    
    for i, s_img in enumerate(section_images):
        q_id = i + 1
        
        # Step 5
        extracted_figures = extract_figures(s_img, q_id)
        
        # Step 6
        question_text = extract_text_without_figures(s_img, extracted_figures)
        
        # Step 7 structure
        q_data = {
            "question_id": q_id,
            "text": question_text,
            "figures": [fig['path'] for fig in extracted_figures]
        }
        dataset.append(q_data)
        
    with open(output_filename, 'w', encoding='utf-8') as f:
        json.dump(dataset, f, indent=4, ensure_ascii=False)
        
    print(f"Successfully saved {len(dataset)} questions to {output_filename}")

In [36]:
# Assuming you already ran the earlier step and generated 'combined_paper.jpg'
results = process_question_paper('combined_paper.jpg')

# Print the structure to verify
with open('data.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=4, ensure_ascii=False)
print("\nExtraction Complete. Data Structure:")
print(json.dumps(results, indent=4))

Splitting into individual questions...
1326
1627
1920
2308
2510
2749
4215
4837
5119
5530
6000
6057
7518
7892
8262
Saved: extracted_data/question_1.jpg
Saved: extracted_data/question_2.jpg
Saved: extracted_data/question_3.jpg
Saved: extracted_data/question_4.jpg
Saved: extracted_data/question_5.jpg
Saved: extracted_data/question_6.jpg
Saved: extracted_data/question_7.jpg
Saved: extracted_data/question_8.jpg
Saved: extracted_data/question_9.jpg
Saved: extracted_data/question_10.jpg
Saved: extracted_data/question_11.jpg
Saved: extracted_data/question_12.jpg
Saved: extracted_data/question_13.jpg
Saved: extracted_data/question_14.jpg
Saved: extracted_data/question_15.jpg

Extraction Complete. Data Structure:
[
    {
        "question_id": 1,
        "question_image_path": "extracted_data/question_1.jpg",
        "extracted_figures": [],
        "question_text": "1. Find the sum of the order and the degree of the differential equation :\n\n2 2\nRUNCR\ndx dx"
    },
    {
        "question_id